In [ ]:
!pip install -q kailash kailash-ml polars torch torchvision matplotlib python-dotenv torch-geometric


# MLFP05 — Exercise 6.2: Graph Attention Network (GAT)


**WHAT YOU'LL LEARN:**
- Why not all neighbours are equally important on a graph
- Attention on graphs: learned edge weights instead of fixed Laplacian
- Build a GAT layer with masked softmax attention over neighbours
- Visualise attention weights to see which citations matter most
- Train a node classifier on Cora and compare to GCN
- Track training with kailash-ml ExperimentTracker


**PREREQUISITES: M5/ex_6.1 (GCN), M5/ex_4 (attention mechanisms).**

**ESTIMATED TIME: ~30 min**


In [ ]:
from __future__ import annotations

import asyncio

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from shared.mlfp05.ex_6 import (
    OUTPUT_DIR,
    device,
    load_graph_data,
    plot_attention_weights,
    plot_graph_with_embeddings,
    plot_node_embeddings,
    plot_training_curves,
    register_model,
    setup_engines,
    train_node_classifier,
)
from kailash_ml.types import MetricSpec

import matplotlib.pyplot as plt


## PHASE 1 — THEORY: Why Attention on Graphs?
GCN treats all neighbours equally — the aggregation weights come from
node degrees (the Laplacian), not from the content of the nodes. But
in real graphs, not all connections carry the same signal:
Example: A paper on "Graph Neural Networks for Drug Discovery" cites:
- 5 foundational GNN papers (highly relevant)
- 3 general drug-discovery reviews (moderately relevant)
- 2 statistics textbooks (barely relevant)
GCN averages ALL 10 citations equally. A human researcher would focus
on the GNN papers. GAT learns to do the same via ATTENTION.
How GAT attention works (for node i attending to neighbour j):
1. Transform both node features: Wh_i, Wh_j
2. Compute attention score: e_ij = LeakyReLU(a^T [Wh_i || Wh_j])
3. Normalise across neighbours: alpha_ij = softmax_j(e_ij)
4. Weighted aggregation: h'_i = sigma( Sum_j alpha_ij * Wh_j )
The attention parameters (a, W) are LEARNED during training, so the
network discovers which neighbour relationships matter for the task.
Think of it as each node having a "spotlight" that it can shine on
its neighbours — brighter light = more attention = more influence
on the node's updated representation.


In [ ]:
print("=" * 70)
print("  PHASE 1 — THEORY: Graph Attention Networks")
print("=" * 70)
print(
    """
  WHY ATTENTION ON GRAPHS?
  GCN limitation: all neighbours contribute equally (degree-based weights).
  In reality, some connections carry more signal than others.

  GAT ATTENTION MECHANISM:
  1. Transform: project both nodes' features through W
  2. Score: e_ij = LeakyReLU( a^T [Wh_i || Wh_j] )
  3. Normalise: alpha_ij = softmax over j in Neighbours(i)
  4. Aggregate: h'_i = sigma( Sum_j alpha_ij * Wh_j )

  Key difference from GCN:
  - GCN: weights are FIXED (come from graph structure / degree)
  - GAT: weights are LEARNED (content-dependent attention)

  Analogy: GCN treats every citation as equally important.
  GAT learns that citing a seminal GNN paper matters more than
  citing a statistics textbook.
"""
)


## PHASE 2 — BUILD: GAT Layer Implementation


In [ ]:
print("=" * 70)
print("  PHASE 2 — BUILD: GAT Layer + Model")
print("=" * 70)

# Load graph data and set up engines
graph_data = load_graph_data()
conn, tracker, exp_name, registry, has_registry = setup_engines()

X = graph_data["X"]
A = graph_data["A"]
y = graph_data["y"]
y_np = graph_data["y_np"]
A_np = graph_data["A_np"]
N = graph_data["N"]
F_dim = graph_data["F_dim"]
n_classes = graph_data["n_classes"]
dataset_name = graph_data["dataset_name"]

HIDDEN_DIM = 16 if dataset_name == "Karate Club" else 64
EPOCHS = 100


class GATLayer(nn.Module):
    """Single GAT layer with masked softmax attention.

    Instead of symmetric-normalised aggregation, GAT learns attention
    weights alpha_ij between each pair of connected nodes:

      e_ij = LeakyReLU( a_src(Wh_i) + a_dst(Wh_j) )
      alpha_ij = softmax_j(e_ij)  over the neighbourhood of i
      h'_i = sigma( Sum_j alpha_ij * Wh_j )

    We compute e_ij for ALL node pairs via broadcasting, then mask out
    non-neighbours with -inf so the softmax ignores them. No per-node loop.
    """

    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        # TODO: Create three linear layers:
        # - self.W: projects features in_dim -> out_dim (no bias)
        # - self.a_src: projects out_dim -> 1 (source attention, no bias)
        # - self.a_dst: projects out_dim -> 1 (destination attention, no bias)
        # Hint: nn.Linear(in_dim, out_dim, bias=False)
        pass
        # Store attention weights for visualisation
        self._alpha: torch.Tensor | None = None

    def forward(self, h: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        # TODO: Implement GAT forward pass:
        # 1. Project features: Wh = self.W(h)  -> shape (N, out_dim)
        # 2. Compute source attention: e_src = self.a_src(Wh)  -> shape (N, 1)
        # 3. Compute dest attention: e_dst = self.a_dst(Wh)  -> shape (N, 1)
        # 4. Broadcast scores: scores = LeakyReLU(e_src + e_dst.T, slope=0.2)  -> (N, N)
        # 5. Mask non-neighbours: add self-loops to adj, set non-edges to -inf
        #    mask = adj + torch.eye(adj.size(0), device=adj.device)
        #    scores = scores.masked_fill(mask == 0, float("-inf"))
        # 6. Normalise: alpha = softmax(scores, dim=1)
        # 7. Cache alpha: self._alpha = alpha.detach()
        # 8. Aggregate: return alpha @ Wh
        pass


class GAT(nn.Module):
    """Two-layer GAT for node classification."""

    def __init__(self, in_dim: int, hidden_dim: int, n_classes: int):
        super().__init__()
        # TODO: Create two GAT layers
        # Layer 1: in_dim -> hidden_dim
        # Layer 2: hidden_dim -> n_classes
        pass

    def forward(self, h: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        # TODO: Two-layer forward: ELU after layer 1, dropout(0.5), then layer 2
        # Hint: Use F.elu (not F.relu) — GAT convention from the original paper
        pass

    def embed(self, h: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        """Return the hidden-layer embedding (before classification head)."""
        # TODO: Return ELU(layer1(h, adj))
        pass

    def get_attention_weights(self) -> tuple[np.ndarray | None, np.ndarray | None]:
        """Return cached attention weights from both layers."""
        alpha1 = self.l1._alpha.cpu().numpy() if self.l1._alpha is not None else None
        alpha2 = self.l2._alpha.cpu().numpy() if self.l2._alpha is not None else None
        return alpha1, alpha2


gat = GAT(in_dim=F_dim, hidden_dim=HIDDEN_DIM, n_classes=n_classes)
n_params = sum(p.numel() for p in gat.parameters())
print(f"\n  GAT architecture:")
print(f"    Layer 1: GATLayer({F_dim} -> {HIDDEN_DIM}) with attention")
print(f"    Layer 2: GATLayer({HIDDEN_DIM} -> {n_classes}) with attention")
print(f"    Total parameters: {n_params:,}")
print(f"\n  How attention works:")
print(f"    For each node i and neighbour j:")
print(f"    1. e_ij = LeakyReLU(a_src(Wh_i) + a_dst(Wh_j))")
print(f"    2. alpha_ij = softmax over all neighbours of i")
print(f"    3. h'_i = ELU(Sum_j alpha_ij * Wh_j)")


### Build Checkpoint ────────────────────────────────────────────────


In [ ]:
assert isinstance(gat, nn.Module), "GAT should be an nn.Module"
assert n_params > 0, "GAT should have learnable parameters"
print("\n--- Build checkpoint passed --- GAT architecture created\n")


## PHASE 3 — TRAIN: Node Classification on Cora


In [ ]:
print("=" * 70)
print(f"  PHASE 3 — TRAIN: GAT on {dataset_name}")
print("=" * 70)

gat_losses, gat_val, gat_test = train_node_classifier(
    model=gat,
    name="GAT",
    forward_arg=A,
    graph_data=graph_data,
    tracker=tracker,
    exp_name=exp_name,
    epochs=EPOCHS,
)


### Train Checkpoint ────────────────────────────────────────────────


In [ ]:
assert len(gat_losses) == EPOCHS, f"Expected {EPOCHS} epoch losses for GAT"
assert gat_losses[-1] < gat_losses[0], "GAT loss should decrease"
best_val = max(gat_val)
best_test = max(gat_test)
print(f"\n  GAT Results:")
print(f"    Best validation accuracy: {best_val:.4f}")
print(f"    Best test accuracy:       {best_test:.4f}")
print(f"    Final loss:               {gat_losses[-1]:.4f}")
# INTERPRETATION: GAT replaces the fixed Laplacian weights with LEARNED
# attention scores. Each node decides how much to attend to each neighbour
# based on the content of both nodes' features. This lets the model
# assign different importance to different neighbours — a citation from
# a highly relevant paper gets more weight than a tangential one.
print("\n--- Train checkpoint passed --- GAT trained successfully\n")


## PHASE 4 — VISUALISE: Attention Weights + Node Embeddings


In [ ]:
print("=" * 70)
print("  PHASE 4 — VISUALISE: GAT Attention Weights + Embeddings")
print("=" * 70)

# Extract embeddings and attention weights
gat.eval()
with torch.no_grad():
    gat_emb = gat.embed(X, A).cpu().numpy()
    # Run a full forward pass to populate attention caches
    _ = gat(X, A)

alpha_l1, alpha_l2 = gat.get_attention_weights()

# Plot 1: Attention weights for a representative node
if alpha_l1 is not None:
    # Find a node with many neighbours for an informative plot
    degrees = A_np.sum(axis=1)
    # Pick a node with above-median degree
    high_degree_nodes = np.where(degrees > np.median(degrees))[0]
    representative_node = int(high_degree_nodes[0]) if len(high_degree_nodes) > 0 else 0

    plot_attention_weights(
        alpha=alpha_l1,
        A_np=A_np,
        labels=y_np,
        title=f"GAT Layer 1 Attention — {dataset_name}",
        filename="gat_attention_weights_l1.png",
        node_idx=representative_node,
        top_k=min(15, int(degrees[representative_node])),
    )

    # Plot attention weights for a second node from a different class
    alt_class = (y_np[representative_node] + 1) % n_classes
    alt_nodes = np.where((y_np == alt_class) & (degrees > np.median(degrees)))[0]
    if len(alt_nodes) > 0:
        plot_attention_weights(
            alpha=alpha_l1,
            A_np=A_np,
            labels=y_np,
            title=f"GAT Attention — Different Class Node",
            filename="gat_attention_weights_alt.png",
            node_idx=int(alt_nodes[0]),
            top_k=min(15, int(degrees[alt_nodes[0]])),
        )

    # TODO: Plot attention weight distribution across all edges
    # 1. Collect attention weights for up to 5000 edges from A_np
    # 2. Create a histogram with 80 bins
    # 3. Add a vertical line at the median
    # 4. Save to OUTPUT_DIR / "gat_attention_distribution.png"
    # Hint: src_idx, dst_idx = np.where(A_np > 0)
    #        edge_attention = [alpha_l1[s, d] for s, d in zip(src_idx[:5000], dst_idx[:5000])]
    edge_attention = []
    src_idx, dst_idx = np.where(A_np > 0)
    for s, d in zip(src_idx[:5000], dst_idx[:5000]):
        edge_attention.append(alpha_l1[s, d])
    edge_attention = np.array(edge_attention)

    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    # TODO: Create histogram of edge_attention with 80 bins, color="steelblue"
    # TODO: Add vertical line at median with color="red", linestyle="--"
    # TODO: Annotate the median value on the plot
    # TODO: Set xlabel="Attention Weight", ylabel="Count", title with dataset_name
    plt.tight_layout()
    filepath = OUTPUT_DIR / "gat_attention_distribution.png"
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {filepath}")

    median_attn = np.median(edge_attention)

print(f"\n  Attention weight analysis:")
if alpha_l1 is not None:
    print(f"    Min attention:    {edge_attention.min():.6f}")
    print(f"    Max attention:    {edge_attention.max():.6f}")
    print(f"    Median attention: {median_attn:.6f}")
    print(f"    Std attention:    {edge_attention.std():.6f}")
    print(f"    -> Wide spread = model differentiates between neighbours")
    print(f"    -> Narrow spread = model treats neighbours more uniformly")

# Plot 3: Node embeddings (2-D PCA)
coords = plot_node_embeddings(
    embeddings=gat_emb,
    labels=y_np,
    n_classes=n_classes,
    title=f"GAT Node Embeddings — {dataset_name}",
    filename="gat_node_embeddings.png",
)

# Plot 4: Graph structure on embedding space
plot_graph_with_embeddings(
    A_np=A_np,
    embeddings_2d=coords,
    labels=y_np,
    n_classes=n_classes,
    title=f"GAT — Graph Structure in Embedding Space ({dataset_name})",
    filename="gat_graph_embeddings.png",
)

# Plot 5: Training curves
plot_training_curves(
    metrics_dict={"GAT train loss": gat_losses},
    title="GAT Training Loss",
    y_label="Cross-Entropy Loss",
    filename="gat_loss_curve.html",
)
plot_training_curves(
    metrics_dict={"GAT val accuracy": gat_val, "GAT test accuracy": gat_test},
    title="GAT Accuracy",
    y_label="Accuracy",
    filename="gat_accuracy_curves.html",
)


### Visualise Checkpoint ────────────────────────────────────────────


In [ ]:
assert gat_emb.shape == (
    N,
    HIDDEN_DIM,
), f"Embedding shape should be ({N}, {HIDDEN_DIM})"
assert alpha_l1 is not None, "Attention weights should be cached"
print("\n--- Visualise checkpoint passed --- GAT attention + embeddings plotted\n")


## PHASE 5 — APPLY: Fraud Detection in Singapore Payment Network


In [ ]:
print("=" * 70)
print("  PHASE 5 — APPLY: Fraud Detection in Payment Networks (PayNow/NETS)")
print("=" * 70)
print(
    """
  SCENARIO: You're building a fraud detection system for a Singapore
  payment network (PayNow, NETS, or a DBS/OCBC/UOB internal network).

  THE GRAPH:
  - Nodes = bank accounts (~100K accounts)
  - Edges = transactions between accounts
  - Node features = account metadata (age, avg balance, transaction freq)
  - Labels = legitimate (0) vs suspicious (1)

  WHY GAT EXCELS AT FRAUD DETECTION:
  Fraudsters don't operate in isolation — they form networks:
  - Mule account A receives funds from compromised account B
  - A splits and forwards to accounts C, D, E
  - C, D, E withdraw at different ATMs

  GAT's attention mechanism reveals WHICH transaction links the model
  considers most suspicious:
  - High attention on A->C edge = "this transfer is a strong fraud signal"
  - Low attention on A->F edge = "this looks like a normal payment"

  This is interpretable AI: the compliance team can see WHY the model
  flagged an account, not just THAT it did.
"""
)

# TODO: Demonstrate attention-based interpretability using Cora as a proxy
# For 5 randomly selected nodes, show attention concentration:
# 1. Pick 5 random nodes using np.random.default_rng(42)
# 2. For each node, get its neighbours from A_np
# 3. Get attention weights from alpha_l1[node, neighbours]
# 4. Compute: max_attn, min_attn, concentration ratio
# 5. Count how many neighbours capture 80% of attention (cumulative sorted weights)
# 6. Count same-class neighbours
# Hint: sorted_weights = np.sort(attn_weights)[::-1]
#        cumsum = np.cumsum(sorted_weights)
#        n_for_80pct = int(np.searchsorted(cumsum, 0.8 * cumsum[-1]) + 1)
print("  Demonstrating attention-based interpretability:")
print("  (Using Cora as proxy — same principle applies to payment graphs)\n")

if alpha_l1 is not None:
    rng = np.random.default_rng(42)
    sample_nodes = rng.choice(N, 5, replace=False)

    for node in sample_nodes:
        neighbours = np.where(A_np[node] > 0)[0]
        if len(neighbours) == 0:
            continue
        # TODO: Extract attention weights and compute statistics
        # attn_weights = alpha_l1[node, neighbours]
        # max_attn = attn_weights.max()
        # min_attn = attn_weights.min()
        # concentration = max_attn / (min_attn + 1e-8)
        # sorted_weights = np.sort(attn_weights)[::-1]
        # cumsum = np.cumsum(sorted_weights)
        # n_for_80pct = int(np.searchsorted(cumsum, 0.8 * cumsum[-1]) + 1)
        # same_class = (y_np[neighbours] == y_np[node]).sum()
        # print(f"    Node {node:4d} (class {y_np[node]}): ...")
        pass

    print(
        """
  FRAUD DETECTION DEPLOYMENT:
  1. Build transaction graph from SWIFT/FAST payment logs
  2. Node features: account age, avg balance, transaction frequency, time patterns
  3. Train GAT on known fraud cases (SAR filings + manual investigations)
  4. For flagged accounts: extract attention weights to show compliance officers
     WHICH transactions triggered the alert — not a black box
  5. Track with ExperimentTracker — retrain monthly as fraud patterns evolve
  6. Attention weight visualisations serve as evidence in regulatory reports
"""
    )

# Register the GAT model
if has_registry:
    version = register_model(
        registry=registry,
        name=f"m5_gat_{dataset_name.lower().replace(' ', '_')}",
        model=gat,
        metrics=[
            MetricSpec(name="best_val_accuracy", value=best_val),
            MetricSpec(name="best_test_accuracy", value=best_test),
            MetricSpec(name="final_loss", value=gat_losses[-1]),
            MetricSpec(name="hidden_dim", value=float(HIDDEN_DIM)),
            MetricSpec(name="epochs", value=float(EPOCHS)),
        ],
    )
    print(f"  Registered GAT: version={version.version}, val_acc={best_val:.4f}")


### Apply Checkpoint ────────────────────────────────────────────────


In [ ]:
assert (
    alpha_l1 is not None
), "Attention weights should be available for interpretability"
print("\n--- Apply checkpoint passed --- Fraud detection scenario demonstrated\n")


## REFLECTION


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED — GAT")
print("=" * 70)
print(
    f"""
  GRAPH ATTENTION NETWORK (Velickovic et al., 2018):
  [x] Learned attention weights: alpha_ij = softmax(LeakyReLU(a^T[Wh_i||Wh_j]))
  [x] Content-dependent aggregation — each node chooses which neighbours matter
  [x] Attention weights are INTERPRETABLE — see which edges the model uses
  [x] Trained on {dataset_name}: {best_val:.1%} val accuracy, {best_test:.1%} test accuracy
  [x] Visualised attention distributions and per-node attention patterns
  [x] Applied to fraud detection: attention reveals suspicious transactions

  GCN vs GAT TRADE-OFF:
  - GCN: simpler, fewer parameters, fixed weights — good for homogeneous graphs
  - GAT: more expressive, content-dependent, interpretable — good when
    edge importance varies and you need to explain model decisions

  KEY INSIGHT: Attention weights are free interpretability. In regulated
  domains (finance, healthcare), the ability to show WHY a model made a
  decision is as important as the decision itself.

  Next: Exercise 6.3 — GraphSAGE: when your graph is too large to fit
  in memory, you need neighbour SAMPLING and INDUCTIVE learning...
"""
)

# Clean up
await conn.close()
